# Warlords Multi-Agent Toernooi
Dit notebook draait de Warlords-omgeving vanuit de Arcade Learning Environment (ALE) met 4 custom agenten.

Dit notebook is ontworpen om te draaien in Google Colab. Als je hem lokaal draait, moet je mogelijk meer libraries installeren en controleren of hun versies compatibel zijn.

## 1 Installeer libraries

Voer eerst de onderstaande codecell uit om de libraries te installeren.

In [1]:
# Install the necessary libraries
!pip install pettingzoo[atari]
!pip install "autorom[accept-rom-license]"
!pip install --find-links dist/ --no-cache-dir AutoROM[accept-rom-license]

  Using cached pettingzoo-1.26.1-py3-none-any.whl.metadata (8.8 kB)
  Using cached gymnasium-1.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached multi_agent_ale_py-0.1.12.tar.gz (552 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached pygame_ce-2.5.7-cp310-cp310-win_amd64.whl.metadata (11 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached farama_notifications-0.0.6-py3-none-any.whl.metadata (729 bytes)
Using cached pettingzoo-1.26.1-py3-none-any.whl (805 kB)
Using cached gymnasium-1.3.0-py3-none-any.whl (953 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached farama_notifications-0.0.6-py3-none-any.whl (2.9 kB)
Using cached pygame_ce-2

  error: subprocess-exited-with-error
  
  × Building wheel for multi_agent_ale_py (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [128 lines of output]
      running bdist_wheel
      running build
      running build_py
      creating build\lib.win-amd64-cpython-310\multi_agent_ale_py
      copying multi_agent_ale_py\ale_python_interface.py -> build\lib.win-amd64-cpython-310\multi_agent_ale_py
      copying multi_agent_ale_py\__init__.py -> build\lib.win-amd64-cpython-310\multi_agent_ale_py
      running egg_info
      writing multi_agent_ale_py.egg-info\PKG-INFO
      writing dependency_links to multi_agent_ale_py.egg-info\dependency_links.txt
      writing requirements to multi_agent_ale_py.egg-info\requires.txt
      writing top-level names to multi_agent_ale_py.egg-info\top_level.txt
      reading manifest file 'multi_agent_ale_py.egg-info\SOURCES.txt'
      reading manifest template 'MANIFEST.in'
      adding license file 'LICENSE.md'
      writing manifest fil

  Using cached AutoROM.accept-rom-license-0.6.1.tar.gz (434 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for AutoROM.accept-rom-license: filename=autorom_accept_rom_license-0.6.1-py3-none-any.whl size=446730 sha256=ad0a97922f8e3ea4043b8f430932825c1ac9991a121b6605f6f6098beb7ab659
  Stored in directory: c:\users\henry\appdata\local\pip\cache\wheels\6b\1b\ef\a43ff1a2f1736d5711faa1ba4c1f61be1131b8899e6a057811
Successfully built AutoROM.accept-rom-license
Looking in links: dist/


**Herstart nu je kernel**. Na het herstarten kun je direct doorgaan met de volgende codecel.

## 2 Importeer libraries en download Atari ROMs

Je krijgt een prompt om de AutoROM-overeenkomst te accepteren. Druk op "Y" wanneer je dit ziet.

In [2]:
# Start AutoROM

!AutoROM

# Import libraries
from pettingzoo.atari import warlords_v3
from pettingzoo.utils import BaseParallelWrapper
import gymnasium as gym
import numpy as np
from collections import defaultdict, Counter
import importlib
import os
import imageio

^C


ModuleNotFoundError: No module named 'multi_agent_ale_py'

## 3 Initialiseer agenten

In deze codecel importeren we de AI-agenten om Warlords te spelen. De bestanden met de agentklassen moeten zich in dezelfde map bevinden als dit notebook.

In [3]:
# Import the agent classes
from agent1 import Agent1
from agent2 import Agent2
from agent3 import Agent3
from agent4 import Agent4

# Instantiate each agent (pass args if needed)
agent_instances = [
    Agent1(),
    Agent2(),
    Agent3(),
    Agent4()
]

agent_names = ['Agent1', 'Agent2', 'Agent3', 'Agent4']
scores = defaultdict(int)
wins = Counter()

# Optioneel: zet deze regel aan om RAM-veranderingen tijdens het spel te printen
# agent_instances = wrap_agents_for_ram_debug(agent_instances, agent_names, print_every=20)


## 4 Speel het spel

## 4a RAM-modus

Deze versie gebruikt **RAM-observaties** in plaats van beeldobservaties.

Bij Atari in PettingZoo geeft `obs_type="ram"` een vector van **128 bytes** terug
(= 1024 bits RAM van de Atari-console). Elke entry in de array is dus één RAM-byte.
De posities van spelers, projectielen en andere game-state zitten **gecodeerd in deze bytes**.

Omdat de exacte byte-indexen voor Warlords game-specifiek zijn, zitten hieronder ook debug-tools
om te zien welke RAM-bytes veranderen tijdens het spel. Daarmee kun je gericht achterhalen welke
bytes de paddleposities, balpositie, etc. voorstellen.

In [4]:
# Hulpfuncties voor RAM-observaties
import numpy as np

def describe_ram_observation(observation):
    observation = np.asarray(observation, dtype=np.uint8)
    print("RAM shape:", observation.shape)
    print("dtype:", observation.dtype)
    print("Eerste 32 bytes:", observation[:32].tolist())

def diff_ram(previous_obs, current_obs):
    previous_obs = np.asarray(previous_obs, dtype=np.uint8)
    current_obs = np.asarray(current_obs, dtype=np.uint8)
    changed = np.where(previous_obs != current_obs)[0]
    return [(int(i), int(previous_obs[i]), int(current_obs[i])) for i in changed]

class RAMDebugAgent:
    """
    Wrapper-agent die elke N stappen RAM-veranderingen print.
    Handig om te reverse-engineeren welke bytes overeenkomen met posities.
    """
    def __init__(self, base_agent, name="Agent", print_every=20):
        self.base_agent = base_agent
        self.name = name
        self.print_every = print_every
        self.prev_obs = None
        self.step_idx = 0

    def act(self, observation):
        observation = np.asarray(observation, dtype=np.uint8)

        if self.step_idx == 0:
            print(f"[{self.name}] start-RAM:")
            describe_ram_observation(observation)
        elif self.step_idx % self.print_every == 0:
            changes = diff_ram(self.prev_obs, observation)
            print(f"[{self.name}] stap {self.step_idx}: {len(changes)} bytes veranderd")
            print("Veranderde bytes (index, oud, nieuw):", changes[:20])

        self.prev_obs = observation.copy()
        self.step_idx += 1
        return self.base_agent.act(observation)

def wrap_agents_for_ram_debug(agent_instances, agent_names, print_every=20):
    return [
        RAMDebugAgent(agent, name=name, print_every=print_every)
        for agent, name in zip(agent_instances, agent_names)
    ]

In deze sectie spelen de vier agenten Warlords. Aan het einde van elk spel wordt de score bijgehouden. De winnaar is de agent met de hoogste score.

In [5]:
# Create environment
env = warlords_v3.env(obs_type="ram", render_mode="rgb_array")

# Prepare directory for videos
video_dir = "./warlords_videos"
os.makedirs(video_dir, exist_ok=True)

print("Observatiemodus:", "RAM (128 bytes per observatie)")


Observatiemodus: RAM (128 bytes per observatie)


De volgende codecel speelt het spel.

In [7]:
# Function to run one game and record video
def run_game(game_id):
    env.reset()

    # Map environment agents to their corresponding agent instances
    agent_mapping = {
        env.agents[i]: agent_instances[i]
        for i in range(len(env.agents))
    }

    # Map environment agents to their corresponding agent names
    agent_name_mapping = {
        env.agents[i]: agent_names[i]
        for i in range(len(env.agents))
    }

    # Reset scores
    for agent in agent_names:
        scores[agent] = 0

    done = False
    terminated = False
    truncated = False

    frames = []
    first_observation_logged = False

    for agent in env.agent_iter():
        observation, reward, termination, truncation, info = env.last()
        scores[agent_name_mapping[agent]] += reward

        if reward > 0:
            print(f"Agent {agent_name_mapping[agent]} won the game!")
            wins[agent_name_mapping[agent]] += 1

        # Use this to debug
        # print(f"Agent: {agent}, Name: {agent_name_mapping[agent]}, Reward: {reward}, Termination: {termination}, Truncation: {truncation}")

        # Deze check hoort binnen de for-loop te vallen
        if not first_observation_logged:
            obs_arr = np.asarray(observation)
            print(f"Eerste observatie voor {agent_name_mapping[agent]}: shape={obs_arr.shape}, dtype={obs_arr.dtype}")
            print("Eerste 32 RAM-bytes:", obs_arr[:32].tolist())
            first_observation_logged = True

        if termination or truncation:
            action = None
        else:
            # this is where you would insert your policy
            agent_obj = agent_mapping[agent]
            action = agent_obj.act(observation)

        env.step(action)

        # Capture the rendered frame
        frame = env.render()
        frames.append(frame)

    env.close()

    # Save video using imageio
    video_path = os.path.join(video_dir, f"game_{game_id}.mp4")
    imageio.mimsave(video_path, frames, fps=15)

In [8]:
# Run 10 games
for game in range(10):
    print(f"Running game {game + 1}...")
    run_game(game_id=game)

print("\nFinal Scores:")
for agent in agent_names:
    print(f"{agent}: Total Reward = {scores[agent]}, Wins = {wins[agent]}")

try:
    winner = wins.most_common(1)[0]
    print(f"Winner: {winner[0]} with {winner[1]} wins!")
except IndexError:
    print("No winners found.")

Running game 1...
Eerste observatie voor Agent1: shape=(128,), dtype=uint8
Eerste 32 RAM-bytes: [0, 0, 33, 109, 79, 36, 107, 5, 66, 60, 60, 0, 31, 31, 239, 240, 240, 240, 240, 240, 240, 0, 0, 255, 255, 255, 255, 255, 255, 31, 31, 255]
Agent Agent4 won the game!


Running game 2...
Eerste observatie voor Agent1: shape=(128,), dtype=uint8
Eerste 32 RAM-bytes: [0, 0, 33, 109, 79, 36, 107, 5, 66, 60, 60, 0, 31, 31, 239, 240, 240, 240, 240, 240, 240, 0, 0, 255, 255, 255, 255, 255, 255, 31, 31, 255]


Agent Agent3 won the game!
Running game 3...
Eerste observatie voor Agent1: shape=(128,), dtype=uint8
Eerste 32 RAM-bytes: [0, 0, 33, 109, 79, 36, 107, 5, 66, 60, 60, 0, 31, 31, 239, 240, 240, 240, 240, 240, 240, 0, 0, 255, 255, 255, 255, 255, 255, 31, 31, 255]


Agent Agent4 won the game!
Running game 4...
Eerste observatie voor Agent1: shape=(128,), dtype=uint8
Eerste 32 RAM-bytes: [0, 0, 33, 109, 79, 36, 107, 5, 66, 60, 60, 0, 31, 31, 239, 240, 240, 240, 240, 240, 240, 0, 0, 255, 255, 255, 255, 255, 255, 31, 31, 255]


Agent Agent2 won the game!
Running game 5...
Eerste observatie voor Agent1: shape=(128,), dtype=uint8
Eerste 32 RAM-bytes: [0, 0, 33, 109, 79, 36, 107, 5, 66, 60, 60, 0, 31, 31, 239, 240, 240, 240, 240, 240, 240, 0, 0, 255, 255, 255, 255, 255, 255, 31, 31, 255]


...

Agent Agent4 won the game!
Running game 7...
Eerste observatie voor Agent1: shape=(128,), dtype=uint8
Eerste 32 RAM-bytes: [0, 0, 33, 109, 79, 36, 107, 5, 66, 60, 60, 0, 31, 31, 239, 240, 240, 240, 240, 240, 240, 0, 0, 255, 255, 255, 255, 255, 255, 31, 31, 255]


Agent Agent1 won the game!
Running game 8...
Eerste observatie voor Agent1: shape=(128,), dtype=uint8
Eerste 32 RAM-bytes: [0, 0, 33, 109, 79, 36, 107, 5, 66, 60, 60, 0, 31, 31, 239, 240, 240, 240, 240, 240, 240, 0, 0, 255, 255, 255, 255, 255, 255, 31, 31, 255]


Agent Agent2 won the game!
Running game 9...
Eerste observatie voor Agent1: shape=(128,), dtype=uint8
Eerste 32 RAM-bytes: [0, 0, 33, 109, 79, 36, 107, 5, 66, 60, 60, 0, 31, 31, 239, 240, 240, 240, 240, 240, 240, 0, 0, 255, 255, 255, 255, 255, 255, 31, 31, 255]


Agent Agent2 won the game!
Running game 10...
Eerste observatie voor Agent1: shape=(128,), dtype=uint8
Eerste 32 RAM-bytes: [0, 0, 33, 109, 79, 36, 107, 5, 66, 60, 60, 0, 31, 31, 239, 240, 240, 240, 240, 240, 240, 0, 0, 255, 255, 255, 255, 255, 255, 31, 31, 255]


Agent Agent2 won the game!

Final Scores:
Agent1: Total Reward = -1, Wins = 1
Agent2: Total Reward = 1, Wins = 4
Agent3: Total Reward = -1, Wins = 2
Agent4: Total Reward = -1, Wins = 3
Winner: Agent2 with 4 wins!


In [9]:
# Display download links for videos
import glob
from IPython.display import FileLink, display
video_files = sorted(glob.glob(f"{video_dir}/*.mp4"))
print("\nDownload the recorded games:")
for file in video_files:
    display(FileLink(file))


Download the recorded games:


/content/warlords_videos/game_0.mp4

/content/warlords_videos/game_1.mp4

/content/warlords_videos/game_2.mp4

/content/warlords_videos/game_3.mp4

...

/content/warlords_videos/game_5.mp4

/content/warlords_videos/game_6.mp4

/content/warlords_videos/game_7.mp4

/content/warlords_videos/game_8.mp4

/content/warlords_videos/game_9.mp4